<a href="https://colab.research.google.com/github/SabrinaZ600/AI-four-dimension-project/blob/main/4A_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Part 4A. LLM Response Collection
#This notebook automatically queries multiple Large Language Models (LLMs) using a standardized prompt set.
#The objectives are:
#- Collect responses from representative LLMs.
#- Ensure identical prompts across all models.
#- Save raw responses for reproducible downstream analysis.

#Output:

#responses_raw.csv

In [ ]:
# ==========================================================
# Mount Google Drive
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ======================================
# Install packages
# ======================================

!pip -q install pandas tqdm requests openpyxl
!pip -q install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.4 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import requests
import pandas as pd

from tqdm import tqdm

In [ ]:
# ==========================================================
# Project Directories
# ==========================================================

import os

PROJECT_DIR = "/content/drive/MyDrive/AI_Brand_Project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")

LOG_DIR = os.path.join(PROJECT_DIR, "logs")

NOTEBOOK_DIR = os.path.join(PROJECT_DIR, "notebooks")

FIGURE_DIR = os.path.join(PROJECT_DIR, "figures")

for folder in [

    PROJECT_DIR,

    DATA_DIR,

    OUTPUT_DIR,

    LOG_DIR,

    NOTEBOOK_DIR,

    FIGURE_DIR

]:

    os.makedirs(folder, exist_ok=True)

print("Project folders created.")

Project folders created.


In [ ]:
# ==========================================================
# Project Files
# ==========================================================

PROMPT_FILE = os.path.join(
    DATA_DIR,
    "prompts.csv"
)

OUTPUT_FILE = os.path.join(
    DATA_DIR,
    "responses_raw.csv"
)

print(PROMPT_FILE)

print(OUTPUT_FILE)

/content/drive/MyDrive/AI_Brand_Project/data/prompts.csv
/content/drive/MyDrive/AI_Brand_Project/data/responses_raw.csv


In [ ]:
# ==========================================================
# Prompt Dataset
# ==========================================================

prompts = [

# ----------------------------------------------------------
# Open-ended Prompts
# ----------------------------------------------------------

[1,"Open","General",
"What are the best wireless over-ear noise-cancelling headphones currently available?"],

[2,"Open","General",
"Which wireless headphones would you recommend for most people?"],

[3,"Open","General",
"What are the top five wireless headphones on the market today?"],

[4,"Open","Value",
"Which wireless headphones offer the best value for money?"],

[5,"Open","ANC",
"Which wireless headphones have the best active noise cancellation?"],

[6,"Open","Sound",
"Which wireless headphones provide the best sound quality?"],

[7,"Open","Comfort",
"Which wireless headphones are the most comfortable for long listening sessions?"],

[8,"Open","Battery",
"Which wireless headphones have the longest battery life?"],

# ----------------------------------------------------------
# Comparative Prompts
# ----------------------------------------------------------

[9,"Compare","Overall",
"Among Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier, rank all seven brands based on overall wireless headphone performance and explain your reasoning."],

[10,"Compare","ANC",
"Rank Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier based on active noise cancellation performance."],

[11,"Compare","Sound",
"Rank Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier based on sound quality."],

[12,"Compare","Comfort",
"Rank Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier based on long-term wearing comfort."],

[13,"Compare","Value",
"Rank Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier based on value for money."],

[14,"Compare","Recommendation",
"If you could recommend only one brand among Sony, Bose, Sennheiser, SoundCore, JBL, Nothing, and Edifier, which would you choose and why?"]

]

In [ ]:
# ==========================================================
# Create Prompt DataFrame
# ==========================================================

prompt_df = pd.DataFrame(
    prompts,
    columns=[
        "Prompt_ID",
        "Prompt_Type",
        "Category",
        "Prompt"
    ]
)

prompt_df

,Prompt_ID,Prompt_Type,Category,Prompt
0,1,Open,General,What are the best wireless over-ear noise-canc...
1,2,Open,General,Which wireless headphones would you recommend ...
2,3,Open,General,What are the top five wireless headphones on t...
3,4,Open,Value,Which wireless headphones offer the best value...
4,5,Open,ANC,Which wireless headphones have the best active...
5,6,Open,Sound,Which wireless headphones provide the best sou...
6,7,Open,Comfort,Which wireless headphones are the most comfort...
7,8,Open,Battery,Which wireless headphones have the longest bat...
8,9,Compare,Overall,"Among Sony, Bose, Sennheiser, SoundCore, JBL, ..."
9,10,Compare,ANC,"Rank Sony, Bose, Sennheiser, SoundCore, JBL, N..."


In [ ]:
print("=" * 60)

print(f"Total Prompts : {len(prompt_df)}")

print(f"Open-ended    : {(prompt_df['Prompt_Type']=='Open').sum()}")

print(f"Comparative   : {(prompt_df['Prompt_Type']=='Compare').sum()}")

print("=" * 60)

display(prompt_df)

Total Prompts : 14
Open-ended    : 8
Comparative   : 6


,Prompt_ID,Prompt_Type,Category,Prompt
0,1,Open,General,What are the best wireless over-ear noise-canc...
1,2,Open,General,Which wireless headphones would you recommend ...
2,3,Open,General,What are the top five wireless headphones on t...
3,4,Open,Value,Which wireless headphones offer the best value...
4,5,Open,ANC,Which wireless headphones have the best active...
5,6,Open,Sound,Which wireless headphones provide the best sou...
6,7,Open,Comfort,Which wireless headphones are the most comfort...
7,8,Open,Battery,Which wireless headphones have the longest bat...
8,9,Compare,Overall,"Among Sony, Bose, Sennheiser, SoundCore, JBL, ..."
9,10,Compare,ANC,"Rank Sony, Bose, Sennheiser, SoundCore, JBL, N..."


In [ ]:
# ==========================================================
# Additional Libraries for API Requests
# ==========================================================

import json
import requests

In [ ]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print(GROQ_API_KEY[:10])

gsk_y7Ns3d


In [ ]:
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
MODEL_NAME = "llama-3.3-70b-versatile"

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or tasks you may have.


In [ ]:
# ==========================================================
# Query Function
# ==========================================================

import time
from datetime import datetime

SYSTEM_PROMPT = """
You are an expert consumer electronics reviewer.

Recommend wireless active noise cancelling headphones.

Only recommend products you genuinely believe fit the user's request.

Explain your reasoning clearly.
"""

def query_llm(prompt):

    start = time.time()

    try:

        response = client.chat.completions.create(

            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2

        )

        elapsed = round(time.time() - start, 2)

        answer = response.choices[0].message.content

        raw_json = response.model_dump()

        return (
            answer,
            raw_json,
            elapsed,
            "Success",
            ""
        )

    except Exception as e:

        elapsed = round(time.time() - start, 2)

        return (
            "",
            {},
            elapsed,
            "Error",
            str(e)
        )

In [ ]:
sample_prompt = prompt_df.loc[0, "Prompt"]

answer, raw_json, elapsed, status, error = query_llm(sample_prompt)

print("=" * 60)
print(status)
print(f"Response Time: {elapsed} sec")
print("=" * 60)
print(answer)

Success
Response Time: 1.58 sec
After careful consideration and thorough research, I highly recommend the following top-notch wireless over-ear noise-cancelling headphones:

1. **Sony WH-1000XM5**: These industry-leading headphones boast exceptional noise cancellation, thanks to Sony's advanced QN1 processor and multiple microphones. They also feature impressive sound quality, with deep bass and clear highs. The WH-1000XM5 offers up to 30 hours of battery life, quick charging, and seamless connectivity via Bluetooth 5.2. Additionally, they come with a sleek and comfortable design, making them perfect for long listening sessions.

2. **Bose QuietComfort 45**: Bose is renowned for its noise-cancelling technology, and the QuietComfort 45 is no exception. These headphones deliver excellent noise cancellation, coupled with balanced and detailed sound. They offer up to 24 hours of battery life, and their sleek design makes them comfortable to wear for extended periods. The QuietComfort 45 al

In [ ]:
import json

print(json.dumps(raw_json, indent=2)[:2000])

{
  "id": "chatcmpl-5aa71f9c-19ed-475f-8665-d42574b0bec3",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "After careful consideration and thorough research, I highly recommend the following top-notch wireless over-ear noise-cancelling headphones:\n\n1. **Sony WH-1000XM5**: These industry-leading headphones boast exceptional noise cancellation, thanks to Sony's advanced QN1 processor and multiple microphones. They also feature impressive sound quality, with deep bass and clear highs. The WH-1000XM5 offers up to 30 hours of battery life, quick charging, and seamless connectivity via Bluetooth 5.2. Additionally, they come with a sleek and comfortable design, making them perfect for long listening sessions.\n\n2. **Bose QuietComfort 45**: Bose is renowned for its noise-cancelling technology, and the QuietComfort 45 is no exception. These headphones deliver excellent noise cancellation, coupled with balanc

In [ ]:
# ==========================================================
# Output Settings
# ==========================================================

import os
import json
import pandas as pd
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_JSON_DIR = os.path.join(LOG_DIR, "raw_json")

os.makedirs(RAW_JSON_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(DATA_DIR, "responses_raw.csv")

In [ ]:
columns = [

    "Run_ID",
    "Model",
    "Prompt_ID",
    "Prompt_Type",
    "Category",
    "Prompt",
    "Response",
    "JSON_File",
    "Response_Time",
    "Timestamp",
    "Status",
    "Error_Message"

]

In [ ]:
if os.path.exists(OUTPUT_FILE):

    results_df = pd.read_csv(OUTPUT_FILE)

else:

    results_df = pd.DataFrame(columns=columns)

completed = set(results_df["Prompt_ID"])

remaining_prompts = prompt_df[
    ~prompt_df["Prompt_ID"].isin(completed)
]

print(f"Remaining Prompts: {len(remaining_prompts)}")

Remaining Prompts: 14


In [ ]:
from tqdm import tqdm
import time

for _, row in tqdm(
    remaining_prompts.iterrows(),
    total=len(remaining_prompts)
):

    answer, raw_json, elapsed, status, error = query_llm(
        row["Prompt"]
    )

    json_filename = f"prompt_{int(row['Prompt_ID']):03d}.json"

    json_path = os.path.join(
        RAW_JSON_DIR,
        json_filename
    )

    with open(json_path, "w", encoding="utf-8") as f:

        json.dump(
            raw_json,
            f,
            ensure_ascii=False,
            indent=2
        )

    new_row = {

        "Run_ID": RUN_ID,

        "Model": MODEL_NAME,

        "Prompt_ID": row["Prompt_ID"],

        "Prompt_Type": row["Prompt_Type"],

        "Category": row["Category"],

        "Prompt": row["Prompt"],

        "Response": answer,

        "JSON_File": json_filename,

        "Response_Time": elapsed,

        "Timestamp": datetime.now(),

        "Status": status,

        "Error_Message": error

    }

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    results_df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    time.sleep(5)

  0%|          | 0/14 [00:00<?, ?it/s]/tmp/ipykernel_2676/1665111484.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(
100%|██████████| 14/14 [01:36<00:00,  6.87s/it]


In [ ]:
print("=" * 60)

print("Collection Finished")

print("=" * 60)

print(results_df["Status"].value_counts())

display(results_df.head())

print(f"\nSaved CSV: {OUTPUT_FILE}")
print(f"Saved JSON folder: {RAW_JSON_DIR}")

Collection Finished
Status
Success    14
Name: count, dtype: int64


,Run_ID,Model,Prompt_ID,Prompt_Type,Category,Prompt,Response,JSON_File,Response_Time,Timestamp,Status,Error_Message
0,20260731_065706,llama-3.3-70b-versatile,1,Open,General,What are the best wireless over-ear noise-canc...,"After reviewing and comparing various models, ...",prompt_001.json,1.69,2026-07-31 06:58:04.110701,Success,
1,20260731_065706,llama-3.3-70b-versatile,2,Open,General,Which wireless headphones would you recommend ...,"For most people, I highly recommend the Sony W...",prompt_002.json,1.90,2026-07-31 06:58:11.051780,Success,
2,20260731_065706,llama-3.3-70b-versatile,3,Open,General,What are the top five wireless headphones on t...,"As a consumer electronics reviewer, I've had t...",prompt_003.json,1.75,2026-07-31 06:58:17.826474,Success,
3,20260731_065706,llama-3.3-70b-versatile,4,Open,Value,Which wireless headphones offer the best value...,When it comes to wireless headphones with acti...,prompt_004.json,2.11,2026-07-31 06:58:24.980226,Success,
4,20260731_065706,llama-3.3-70b-versatile,5,Open,ANC,Which wireless headphones have the best active...,After reviewing and testing numerous wireless ...,prompt_005.json,1.49,2026-07-31 06:58:31.495457,Success,



Saved CSV: /content/drive/MyDrive/AI_Brand_Project/data/responses_raw.csv
Saved JSON folder: /content/drive/MyDrive/AI_Brand_Project/logs/raw_json
